In [ ]:
'Final dataframe using the prominence method and the negative LAP data filter'

'CHECKED'

In [1]:
import datetime 
import numpy as np
import mpmath as mp
import pandas as pd
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import copy
from sklearn.preprocessing import StandardScaler
%matplotlib qt

import os 
from os import listdir
from os.path import isfile, join
#------------------------------------------------------------------------------------
#This function receives a list with magnetic fields data and returns a list with the values of deltaB/B

def newdeltas(magneticfieldcolumn):
    rolling4=magneticfieldcolumn.rolling(600,center=True).mean() #moving average
    element=[] 
    for index1 in np.arange(0,len(rolling4)): #for every element in the rolling average
        if rolling4[index1]==np.nan:
            element.append(np.nan)
        else:    
            delta=((2*(magneticfieldcolumn[index1]-rolling4[index1]))/rolling4[index1])
            element.append(abs(delta))
            
    return element
#-----------------------------------------------------------------------------------------
#PCAnew returns the sorted eigenvalues and eigenvectors of the covariance matrix calculated with the X data 

def PCAnew(X):
    
    #The covariance matrix does not work with NaN values
    #------------------------------------------------
    check_for_nan_1 = X['Bx'].isnull()
    data_copy = X.copy()
    for i in np.arange(0,len(X)):
        if check_for_nan_1[i]==True:
            data_copy.drop(i, axis=0, inplace=True)
    #After this we droped all the nan values in the specific interval
    #-----------------------------------------------------------------------------
    if len(data_copy)!=1 and len(data_copy)!=2 and len(data_copy)!=3 and X.dropna().empty==False: #If the interval does not contain nan numbers
    
        x_std = StandardScaler().fit_transform(data_copy)
        #------------------------------------------------
    
        # features are columns from x_std
        features = x_std.T 
        covariance_matrix = np.cov(features)
        print(covariance_matrix)
        #Eigen Vectors and Eigen Values from Covariance Matrix
        eig_vals, eig_vecs = np.linalg.eig(covariance_matrix)
    
        #In order to print in descending order the vectors...
        vec=eig_vecs.T
        sortedd=sorted(eig_vals, reverse=True)
        neww=[]
    
        for index1 in np.arange(0,3):  
             for index2 in np.arange(0,3):
                if eig_vals[index2]==sortedd[index1]:
                    neww.append(vec[index2])
    else:
        sortedd=[np.nan, np.nan, np.nan]
        neww=[[np.nan, np.nan, np.nan],[np.nan, np.nan, np.nan],[np.nan, np.nan, np.nan]]

    return sortedd, neww  #In order to compute easier it is used the transpose of the eig_vecs 

#-------------------------------------------------------------------------------------------
#Angles between the vectors

import numpy as np
import numpy.linalg as LA

def angles(a,b): #Returns the angle in degrees and radians
    inner = np.inner(a, b)
    norms = LA.norm(a) * LA.norm(b)

    cos = inner / norms
    rad = np.arccos(np.clip(cos, -1.0, 1.0))
    deg = np.rad2deg(rad)
    
    return deg, rad

#-----------------------------------------------------------------------------------------
#bgmgvector is a list with the background magnetic fields vector
  
#The method consist of an iterating decomposing of the data matrix second by second
#For example if the time used is 3 minutes-->180 seconds. The method will use 90 seconds to the left and 90 seconds to the right (similar to the moving average)
#It should be noted that in this example the first 90 and last 90 points will not have any value 


#This function returns a list of lists that contains the eigenvalues ratios of the PCA method, 
#the angles phi and theta and the biggest deltaB/B value for each iteration

def function1(X,n, newdeltaB, bgmgvector): #n must be an even integer
    element=[]
    nan=[np.nan, np.nan, np.nan, np.nan, np.nan]
    L=X.shape[0] #Size of the data
    for i in np.arange(0,L): #For each data set 
        #The first two ifs are going to tell us if the interval exists and allows the iteration to work, based on the quantity of points 
        if i<(n/2):
            element.append(nan)
        elif (L-i)<=(n/2):
            element.append(nan) 
        else: #If the interval exists
            newdata=X.iloc[np.arange(i-n/2,i+1+n/2),:] #Select the interval
            newdataf=newdata.reset_index(drop=True) 
            newdata2=list(newdeltaB[i] for i in np.arange(int(i-n/2),int(i+1+n/2)))
            method=PCA2(newdataf) #PCA using the data of the interval (this fuction returns the eigenvalues and eigenvectors)
            
            if np.isnan(method[0][0])==False and np.isnan(method[1][0][0])==False: #If the PCA exist
                eigenvector1=method[1][0] #eigenvector associated with the biggest eigenvalue 
                eigenvector2=method[1][2] #eigenvector associated with the smallest eigenvalue
                ratio1=method[0][0]/method[0][1] #Goodness betwen the biggest and the middle eigenvalue
                ratio2=method[0][1]/method[0][2] #Goodness betwen the middle and smallest eigenvalue
                mgvector=bgmgvector[i] #each background magnetic field vector
                theta=angles(eigenvector2,mgvector)[0] #angle in degrees associated with the smallest eigenvalue
                phi=angles(eigenvector1,mgvector)[0] #angle in degrees associated with the biggest eigenvalue
                #If the angle is needed in radians change angles[0] to angles [1]
                
                array=[ratio1, ratio2, theta,phi, max(newdata2)]  ##### MAYBE THIS IS NOT WORKING
                #array=[ratio1, ratio2, theta,phi, newdeltaB[i]]
                element.append(array)
            else:
                element.append(nan)
    return element

#------------------------------------------------------------------------------------------------------
def getting_day(data_plasma, year, month, day):
    
    time_plasma=pd.to_datetime(data_plasma['t_utc'])
    #year=2015
    mask = time_plasma.dt.year == int(year)
    include = data_plasma[mask]
    time_plasma2=pd.to_datetime(include['t_utc'])
    #month='06'
    mask2=time_plasma2.dt.month == int(month)
    include2 = include[mask2]
    time_plasma3=pd.to_datetime(include2['t_utc'])
    #day='07'
    mask3=time_plasma3.dt.day == int(day)
    include3=include2[mask3]

    time_plasma4=pd.to_datetime(include3['t_utc'])
    
  
    return time_plasma4, include3['npl']
#------------------------------------------------------------------------------------------------------
def get_directory(folder):
    files= [a for a in listdir(folder) if isfile(join(folder,a))]
    
    return files

#------------------------------------------------------------------------------------------------------
#This function receives a Mirror Mode Waves Candidates DataFrame and returns its time intervals and how many MM are in that day

def intervals(element, n, array1):  
    #modifying the MM table
    element['tvalue'] = element.index
    element['delta'] = (element['tvalue']-element['tvalue'].shift()).fillna(0)
    zx=element.copy()
    zx2=zx.reset_index(drop=True)
    deltas=zx2['delta']
    deltas=deltas.values.tolist()
    indexA=zx2['Index']
    
    #Empty Dataframes
    my_df=pd.DataFrame()
    my_df2=pd.DataFrame()
    
    if len(element['tvalue'])!=0: #If we have MM waves
    
        #LIMIT CONDITIONS
        limits=[]
        limits2=[]
        for i in np.arange(0,len(zx2)):
            if deltas[i]>(n/2) or deltas[i]==0: #Same event
                limits.append(indexA[i])
                if i!=0:
                    limits2.append(indexA[i-1])
        limits2.append(indexA[len(deltas)-1])   
        
    
        if len(limits)!=1: #If there is not only one MM
    
            for i in np.arange(0,len(limits)): 
                index1=limits[i]
                index2=limits2[i]
                my_df=my_df.append({'Beginning' : array1[index1-(n/2)], 'End' : array1[index2+(n/2)], 'Index1' : index1, 'Index2': index2}, ignore_index=True) 
    
        else: #If there is only one MM
            index1=limits[0]
            index2=limits2[0]
            large=len(deltas)
            my_df=my_df.append({'Beginning' : array1[index1-(n/2)], 'End' : array1[index2+(n/2)], 'Index1' : index1, 'Index2': index2}, ignore_index=True)
    else: #If we have not MM
        my_df=my_df
        limits=[]
        limits2=[]


    return my_df, len(my_df), limits,limits2,


def PCA2(X):
    
    #The covariance matrix does not work with NaN values
    #------------------------------------------------
    #print(X)
    check_for_nan_1 = X['Bx'].isnull()
    check_for_nan_2 = X['By'].isnull()
    check_for_nan_3 = X['Bz'].isnull()
    data_copy = X.copy()
    for i in np.arange(0,len(X)):
        if check_for_nan_1[i]==True or check_for_nan_2[i]==True or check_for_nan_3[i]==True :
            data_copy.drop(i, axis=0, inplace=True)
    #After this we droped all the nan values in the specific interval
    #print(data_copy)
    #-----------------------------------------------------------------------------
    if len(data_copy)!=1 and len(data_copy)!=2 and len(data_copy)!=3 and X.dropna().empty==False: #If the interval does not contain nan numbers
    
        covariance_matrix=data_copy.cov()
        #print(covariance_matrix)
        #Eigen Vectors and Eigen Values from Covariance Matrix
        eig_vals, eig_vecs = np.linalg.eig(covariance_matrix)
    
        #In order to print in descending order the vectors...
        vec=eig_vecs.T
        sortedd=sorted(eig_vals, reverse=True)
        neww=[]
    
        for index1 in np.arange(0,3):  
             for index2 in np.arange(0,3):
                if eig_vals[index2]==sortedd[index1]:
                    neww.append(vec[index2])
    else:
        sortedd=[np.nan, np.nan, np.nan]
        neww=[[np.nan, np.nan, np.nan],[np.nan, np.nan, np.nan],[np.nan, np.nan, np.nan]]

    return sortedd, neww  #In order to compute easier it is used the transpose of the eig_vecs 

#---------------------------------------------------------------------------------------------------------------

In [43]:
#IMPORT DATA
'You only need to change this for every year'
#------------------------------------------------------------------------------------------
folder="C:/Users/Ariel/Desktop/JUPYTER/Resultados/peakswithoutnegativevaluesnpl/aa2016"
#------------------------------------------------------------------------------------------
directory= os.scandir(folder)
list_of_files=get_directory(folder)

newlist=[]
for item in list_of_files:
    newlist.append(folder+'/'+str(item))
#print(newlist)

mag_table=[]
ses=[]

titles=['Beginning','End','Index1','Index2','Pearson']
#This stays fixed
data0=pd.read_table("C:/Users/Ariel/Desktop/JUPYTER/Resultados/peakswithoutnegativevaluesnpl/aa2014/MM_final_table_with_pearson_prominence_method_20141102"
                    , header=None, names=titles, sep='\t')

for j in np.arange(0,len(newlist)):
#for j in np.arange(0,1):
    #print(j)
    #Importing data
    titles=['Beginning','End','Index1','Index2','Pearson']
    data= pd.read_table(str(newlist[j]), header=None, names=titles, sep='\t')
    data=data.iloc[np.arange(1,len(data)),:] #delete the first row
    data=data.reset_index(drop=True)
    
    data0 = pd.concat([data0,data], axis=0)
#print(data)
data0=data0.reset_index(drop=True)
print(data0)

               Beginning                  End   Index1   Index2  \
0    2016-01-08 20:41:55  2016-01-08 20:42:27  74530.0  74532.0   
1    2016-01-08 20:46:43  2016-01-08 20:47:18  74818.0  74823.0   
2    2016-01-09 08:24:02  2016-01-09 08:24:32  30257.0  30257.0   
3    2016-01-09 08:56:13  2016-01-09 08:56:47  32188.0  32192.0   
4    2016-01-09 09:37:22  2016-01-09 09:37:54  34657.0  34659.0   
..                   ...                  ...      ...      ...   
100  2016-02-28 00:22:41  2016-02-28 00:23:21   1376.0   1386.0   
101  2016-02-28 06:53:13  2016-02-28 06:53:48  24808.0  24813.0   
102  2016-02-28 11:27:40  2016-02-28 11:28:13  41275.0  41278.0   
103  2016-02-28 11:33:27  2016-02-28 11:34:02  41622.0  41627.0   
104  2016-02-28 11:33:48  2016-02-28 11:34:20  41643.0  41645.0   

                 Pearson  
0    -0.8890605847675127  
1    -0.9075383212767103  
2    -0.7359259211219329  
3    -0.7829051076174524  
4    -0.9145761721502474  
..                   ...  
100  -

In [44]:
import csv
data0.to_csv('zxc2016', index=False, sep='\t') 

In [52]:
'Concatenate the 3 years'

data2014path="C:/Users/Ariel/Desktop/JUPYTER/zxc2014"
data2015path="C:/Users/Ariel/Desktop/JUPYTER/zxc2015"
data2016path="C:/Users/Ariel/Desktop/JUPYTER/zxc2016"

titles=['Beginning','End','Index1','Index2','Pearson']

data2014=pd.read_table(data2014path, header=None, names=titles, sep='\t')
data2014=data2014.iloc[np.arange(1,len(data2014)),:] #delete the first row
data2014=data2014.reset_index(drop=True)

data2015=pd.read_table(data2015path, header=None, names=titles, sep='\t')
data2015=data2015.iloc[np.arange(1,len(data2015)),:] #delete the first row
data2015=data2015.reset_index(drop=True)

data2016=pd.read_table(data2016path, header=None, names=titles, sep='\t')
data2016=data2016.iloc[np.arange(1,len(data2016)),:] #delete the first row
data2016=data2016.reset_index(drop=True)

a1= pd.concat([data2014,data2015], axis=0)
a2= pd.concat([a1,data2016], axis=0)
a2=a2.reset_index(drop=True)
print(a2)

import csv
a2.to_csv('all_prominence_events7', index=False, sep='\t') 

               Beginning                  End   Index1   Index2  \
0    2014-12-14 09:55:30  2014-12-14 09:56:07  35745.0  35752.0   
1    2014-12-14 10:39:07  2014-12-14 10:39:38  38362.0  38363.0   
2    2014-12-15 18:56:06  2014-12-15 18:56:36  68181.0  68181.0   
3    2014-12-18 22:14:20  2014-12-18 22:15:00  80075.0  80085.0   
4    2014-12-24 01:04:58  2014-12-24 01:05:30   3913.0   3915.0   
..                   ...                  ...      ...      ...   
563  2016-02-28 00:22:41  2016-02-28 00:23:21   1376.0   1386.0   
564  2016-02-28 06:53:13  2016-02-28 06:53:48  24808.0  24813.0   
565  2016-02-28 11:27:40  2016-02-28 11:28:13  41275.0  41278.0   
566  2016-02-28 11:33:27  2016-02-28 11:34:02  41622.0  41627.0   
567  2016-02-28 11:33:48  2016-02-28 11:34:20  41643.0  41645.0   

                 Pearson  
0    -0.7731610082993551  
1    -0.7194916719216243  
2    -0.7126112073191414  
3    -0.7372360235593105  
4    -0.8271626975988577  
..                   ...  
563  -

In [ ]:
'Now we have to delete the candidates that does not satisfies the LAP negative values criteria using the dataframe 6 and 7'

In [62]:
'DATAFRAMES'

dataframe6path="C:/Users/Ariel/Desktop/JUPYTER/Resultados/histograms/data_used_for_the_histograms/All_prominence_events6"
dataframe7path="C:/Users/Ariel/Desktop/JUPYTER/Resultados/histograms/data_used_for_the_histograms/All_prominence_events7"

titles1=['Beginning','End','Index1','Index2','Pearson','gas_production_rates','bmoduluspeaks','cometo-centric distance','x','roots','bmodulusaverage']
titles2=['Beginning','End','Index1','Index2','Pearson']
dataframe6=pd.read_table(dataframe6path, header=None, names=titles1, sep='\t')
dataframe6=dataframe6.iloc[np.arange(1,len(dataframe6)),:] #delete the first row
dataframe6=dataframe6.reset_index(drop=True)

dataframe7=pd.read_table(dataframe7path, header=None, names=titles2, sep='\t')
dataframe7=dataframe7.iloc[np.arange(1,len(dataframe7)),:] #delete the first row
dataframe7=dataframe7.reset_index(drop=True)


In [63]:
print('dataframe6')
print(dataframe6)

print('dataframe7')
print(dataframe7)

dataframe6
               Beginning                  End   Index1   Index2  \
0    2014-12-14 09:55:30  2014-12-14 09:56:07  35745.0  35752.0   
1    2014-12-14 10:39:07  2014-12-14 10:39:38  38362.0  38363.0   
2    2014-12-15 18:56:06  2014-12-15 18:56:36  68181.0  68181.0   
3    2014-12-18 22:14:20  2014-12-18 22:15:00  80075.0  80085.0   
4    2014-12-24 01:04:58  2014-12-24 01:05:30   3913.0   3915.0   
..                   ...                  ...      ...      ...   
619  2016-02-28 00:22:41  2016-02-28 00:23:21   1376.0   1386.0   
620  2016-02-28 06:53:13  2016-02-28 06:53:48  24808.0  24813.0   
621  2016-02-28 11:27:40  2016-02-28 11:28:13  41275.0  41278.0   
622  2016-02-28 11:33:27  2016-02-28 11:34:02  41622.0  41627.0   
623  2016-02-28 11:33:48  2016-02-28 11:34:20  41643.0  41645.0   

                 Pearson gas_production_rates       bmoduluspeaks  \
0    -0.7731610082993551           7.5722e+25  6.0298424523365455   
1    -0.7194916719216243          8.53915e+25 

In [65]:

path_time= pd.to_datetime(dataframe6['Beginning']) 
path_time2= pd.to_datetime(dataframe6['End']) 
q=path_time.dt.year
qq=path_time.dt.month
qqq=path_time.dt.day


bmodulus_list=[]
distance_list=[]

#DATA2

for i in np.arange(0,len(dataframe6)): #PARA CADA EVENTO DE LOS 624
#for i in np.arange(1,3):
    A=0
    for j in np.arange(0,len(dataframe7)): #PARA CADA EVENTO DE LOS 568
        if dataframe7['Beginning'][j]==dataframe6['Beginning'][i] and dataframe7['End'][j]==dataframe6['End'][i]:
                
            A=1         
            break
            
    if A==0:
        dataframe6['Beginning'][i]=np.nan
        dataframe6['End'][i]=np.nan
        
#---------------------------------------------  
#DROPPING CANDIDATES THAT HAS NAN VALUES
    
for j in np.arange(0,len(dataframe6)):
    if pd.isnull(dataframe6['Beginning'][j])==True and pd.isnull(dataframe6['End'][j])==True:
        dataframe6.drop(j, axis=0, inplace=True)
        
dataframe6=dataframe6.reset_index(drop=True)        
        
print(dataframe6)        

               Beginning                  End   Index1   Index2  \
0    2014-12-14 09:55:30  2014-12-14 09:56:07  35745.0  35752.0   
1    2014-12-14 10:39:07  2014-12-14 10:39:38  38362.0  38363.0   
2    2014-12-15 18:56:06  2014-12-15 18:56:36  68181.0  68181.0   
3    2014-12-18 22:14:20  2014-12-18 22:15:00  80075.0  80085.0   
4    2014-12-24 01:04:58  2014-12-24 01:05:30   3913.0   3915.0   
..                   ...                  ...      ...      ...   
563  2016-02-28 00:22:41  2016-02-28 00:23:21   1376.0   1386.0   
564  2016-02-28 06:53:13  2016-02-28 06:53:48  24808.0  24813.0   
565  2016-02-28 11:27:40  2016-02-28 11:28:13  41275.0  41278.0   
566  2016-02-28 11:33:27  2016-02-28 11:34:02  41622.0  41627.0   
567  2016-02-28 11:33:48  2016-02-28 11:34:20  41643.0  41645.0   

                 Pearson gas_production_rates       bmoduluspeaks  \
0    -0.7731610082993551           7.5722e+25  6.0298424523365455   
1    -0.7194916719216243          8.53915e+25   16.382777

In [66]:
import csv
dataframe6.to_csv('all_prominence_events8', index=False, sep='\t') 